In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class LensingDataset(Dataset):
    def __init__(self, lenses_dir, nonlenses_dir, transform=None):
        self.filepaths = []
        self.labels = []
        self.transform = transform
        
        for file in os.listdir(lenses_dir):
            if file.endswith('.npy'):
                self.filepaths.append(os.path.join(lenses_dir, file))
                self.labels.append(1) 
                
        
        for file in os.listdir(nonlenses_dir):
            if file.endswith('.npy'):
                self.filepaths.append(os.path.join(nonlenses_dir, file))
                self.labels.append(0)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_array = np.load(self.filepaths[idx])
        img_array = (img_array - np.min(img_array)) / (np.max(img_array) - np.min(img_array) + 1e-8)

        img_tensor = torch.tensor(img_array, dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        if self.transform:
            img_tensor = self.transform(img_tensor)
            
        return img_tensor, label

In [ ]:

transform_train = transforms.Compose([
    transforms.Resize((128, 128), antialias=True), # Upscale for ResNet
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5)
])


transform_test = transforms.Compose([
    transforms.Resize((128, 128), antialias=True)
])


TRAIN_LENSES_DIR = 'dataset2/train_lenses'
TRAIN_NONLENSES_DIR = 'dataset2/train_nonlenses'
TEST_LENSES_DIR = 'dataset2/test_lenses'
TEST_NONLENSES_DIR = 'dataset2/test_nonlenses'

train_dataset = LensingDataset(TRAIN_LENSES_DIR, TRAIN_NONLENSES_DIR, transform=transform_train)
test_dataset = LensingDataset(TEST_LENSES_DIR, TEST_NONLENSES_DIR, transform=transform_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"📦 Loaded {len(train_dataset)} training images and {len(test_dataset)} testing images.")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models


device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2) 
model = model.to(device)

num_lenses = sum(1 for label in train_dataset.labels if label == 1)
num_nonlenses = sum(1 for label in train_dataset.labels if label == 0)

weight_nonlens = 1.0
weight_lens = num_nonlenses / (num_lenses + 1e-8) 

class_weights = torch.tensor([weight_nonlens, weight_lens], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

print(f"⚖️ Class Weights Configured -> Non-Lenses: {weight_nonlens:.2f}, Lenses: {weight_lens:.2f}")

In [ ]:
epochs = 5 

print("🚀 Starting the training process...")
for epoch in range(epochs):
    model.train() 
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
    
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

model.eval() 

all_labels = []
all_probs = []

with torch.no_grad(): 
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)


fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)


plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic - Lenses vs Non-Lenses')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()